# Training Arabic Sentiment Model

Phase 1 — Download & Understand the Dataset

Source: [Arabic Customer Reviews (Kaggle)](https://www.kaggle.com/datasets/mohamedramadan2040/arabic-customer-reviews)

Goal of this notebook (this phase): **understand the data only**. No model training yet.

## 1. Dataset Loading

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 140)

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA_PATH = ROOT / "data" / "raw" / "arabic" / "Final_Data.csv"
print("Loading:", DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH)
df.head()

Loading: /home/abdlrhman/Courses/IEEE SSCS/NLP/Week1/Project1/data/raw/arabic/Final_Data.csv


,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشكل اسرع,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكون فاتحه بقاله كده اكتر من شهر,negative,talbat


## 2. Dataset Inspection

In [2]:
print("shape:", df.shape)
print("n_samples:", df.shape[0])
print("n_columns:", df.shape[1])
print("columns:", list(df.columns))

shape: (40046, 3)
n_samples: 40046
n_columns: 3
columns: ['review_description', 'rating', 'company']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   review_description  40045 non-null  str  
 1   rating              40046 non-null  str  
 2   company             40046 non-null  str  
dtypes: str(3)
memory usage: 5.0 MB


In [4]:
print("text column  : review_description")
print("label column : rating")
print("extra column : company  (review source / app)")
print()
print("unique companies:")
print(df["company"].value_counts())

text column  : review_description
label column : rating
extra column : company  (review source / app)

unique companies:
company
talbat           32073
swvl              4693
telecom_egypt     2090
venus              281
Raya               268
TMG                250
elsewedy           147
hilton             100
capiter             73
Ezz Steel           49
nestle              18
domty                4
Name: count, dtype: int64


## 3. Missing Values

In [5]:
print(df.isnull().sum())
print()
print("rows with any null:", int(df.isnull().any(axis=1).sum()))
print("empty/whitespace text:", int((df["review_description"].fillna("").str.strip() == "").sum()))
df[df["review_description"].isna()]

review_description    1
rating                0
company               0
dtype: int64

rows with any null: 1
empty/whitespace text: 1


,review_description,rating,company
5911,NaN,negative,talbat


## 4. Duplicates

In [6]:
n_full_dups = int(df.duplicated().sum())
n_text_dups = int(df.duplicated(subset=["review_description"]).sum())

print("full-row duplicates :", n_full_dups)
print("duplicate texts     :", n_text_dups)

conflicting = (
    df.dropna(subset=["review_description"])
    .groupby("review_description")["rating"]
    .nunique()
)
n_conflict = int((conflicting > 1).sum())
print("same text, different labels:", n_conflict)

df[df.duplicated(keep=False)].head(10)

full-row duplicates : 28
duplicate texts     : 1042


same text, different labels: 95


,review_description,rating,company
901,حلو جدا,positive,talbat
7243,حلو جدا,positive,talbat
32123,كباتن في غاية قله الادب خدمه عملاء مابتفهمش برنامج كله اصلا ما يستاهل التحميل او التجربه من أسواء برامج التوصيل تم التواصل كذا مرة مع خدمه العملاء ومافيش اي...,negative,swvl
32152,كويس,positive,swvl
32197,جيد,positive,swvl
32452,خدمه مميزه,positive,swvl
32597,سيئ,negative,swvl
32771,شركه محترمه,positive,swvl
33826,حراميه,negative,swvl
34757,معقول,neutral,swvl


## 5. Class Distribution

In [7]:
print("unique labels:", df["rating"].unique().tolist())
print()
print(df["rating"].value_counts())
print()
print((df["rating"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")

unique labels: ['positive', 'negative', 'neutral']

rating
positive    23921
negative    14200
neutral      1925
Name: count, dtype: int64

rating
positive    59.73%
negative    35.46%
neutral      4.81%
Name: proportion, dtype: str


In [8]:
counts = df["rating"].value_counts().reindex(["positive", "negative", "neutral"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="bar", ax=axes[0], color=["#2ca02c", "#d62728", "#7f7f7f"], rot=0)
axes[0].set_title("Arabic class counts")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("count")

counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", colors=["#2ca02c", "#d62728", "#7f7f7f"])
axes[1].set_title("Arabic class share")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

/tmp/ipykernel_28851/3710070145.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
print("Sample review per class:\n")
for label in ["positive", "negative", "neutral"]:
    sample = df.loc[df["rating"] == label, "review_description"].dropna().iloc[0]
    print(f"[{label}] {sample}\n")

Sample review per class:

[positive] رائع

[negative] التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال بالشبكة..مع انه النت عندي تمام شو الحل??

[neutral] تجربه مرضيه والتوصيل كان كويس ومعقول



## Checkpoint — what we know before modeling

| Question | Answer |
|---|---|
| Text column | `review_description` |
| Label column | `rating` |
| Extra column | `company` (Talabat, Swvl, Telecom Egypt, …) |
| Samples | **40,046** rows × 3 columns |
| Nulls | **Yes** — 1 missing `review_description` (label=`negative`, company=`talbat`) |
| Duplicates | **Yes** — 28 full-row duplicates; 1,042 duplicate texts; **95 texts have conflicting labels** |
| Balanced? | **No** |
| Label shape | lowercase strings: `positive` / `negative` / `neutral` |
| Neutral? | **Yes** — Arabic is **3-class**, not binary |

Class mix:

- `positive`: 23,921 (59.73%)
- `negative`: 14,200 (35.46%)
- `neutral`: 1,925 (4.81%) — rare class

**Implications for later phases (do not train yet):**

1. Drop the 1 null-text row.
2. Decide how to handle duplicate texts, especially the 95 with conflicting labels.
3. Neutral is a real class (~5%). This dataset is **not** Positive/Negative only.
4. English IMDb is binary with capitalized labels (`Positive`/`Negative`). Label names and class count must be aligned later in the app.
5. Reviews are short (median ~28 characters) vs long IMDb reviews — preprocessing should not assume the same text length.